# Anti-DPO: Google Colab preparation and audit

The notebook clones the public experiment repository, reads the committed local dataset, validates it, and creates anti-DPO splits. The objective intentionally maps source `rejected` to the preferred completion.

In [ ]:
!nvidia-smi

In [ ]:
# Colab setup: run this cell first.
REPO_URL = 'https://github.com/moliksq/Mauvais.git'
REPO_DIR = '/content/Mauvais'
!rm -rf $REPO_DIR
!git clone --depth 1 $REPO_URL $REPO_DIR
%cd $REPO_DIR/anti_dpo_experiment
!pip -q install 'datasets>=3.0' 'matplotlib>=3.8' 'pandas>=2.0'

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
SOURCE = ROOT.parent / 'train.jsonl'
assert SOURCE.is_file(), f'Missing committed dataset: {SOURCE}'
print(f'Dataset: {SOURCE} ({SOURCE.stat().st_size / 2**20:.2f} MiB)')

In [ ]:
from anti_preference import load_jsonl_pairs, invert_preference_pairs

rows, report = load_jsonl_pairs(SOURCE)
anti_rows = invert_preference_pairs(rows, penalty_strength=.35, max_weight=2.0)
print(report.to_dict())
anti_rows[0]

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].hist([len(row['chosen']) for row in rows], bins=60, range=(0, 2500), alpha=.7, label='source chosen')
axes[0].hist([len(row['rejected']) for row in rows], bins=60, range=(0, 2500), alpha=.7, label='source rejected')
axes[0].set(xlabel='Characters', ylabel='Rows', title='Response lengths')
axes[0].legend()
axes[1].hist([row['anti_weight'] for row in anti_rows], bins=30, color='#9c2c77')
axes[1].set(xlabel='Loss multiplier', ylabel='Rows', title='Bounded anti-DPO weights')
fig.tight_layout()

In [ ]:
from datasets import Dataset, DatasetDict
import json

output = ROOT / 'data' / 'russian_qa'
report_path = ROOT / 'reports' / 'preparation.json'
splits = Dataset.from_list(anti_rows).train_test_split(test_size=.05, seed=42)
DatasetDict(splits).save_to_disk(str(output))
payload = report.to_dict() | {'objective': 'anti_dpo', 'preference_mapping': 'source.rejected -> chosen; source.chosen -> rejected', 'penalty_strength': .35, 'max_weight': 2.0, 'train_rows': len(splits['train']), 'test_rows': len(splits['test'])}
report_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'Saved {len(splits["train"])} train and {len(splits["test"])} test rows.')